# BIT Change Detection - Foundation-Model Backbone Comparison

Swaps BIT's ImageNet ResNet backbone for two foundation-model encoders and compares them on LEVIR-CD (in-domain) and WHU-CD (zero-shot):

- `bit` - BIT + DOFA, an Earth-observation foundation model.
- `bit_dino` - BIT + DINOv2, a general-purpose vision foundation model.

This is a cleaned research notebook. It keeps the useful experiment steps and removes repeated cells caused by Colab session resets.


## 0. Setup — run once per fresh runtime


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

import os
!git clone -q https://github.com/zhu-xlab/DOFA.git /content/DOFA
!git clone -q https://github.com/rushammm/BIT_CD.git /content/BIT_CD

# DOFA pretrained weights: download once, then cache on Drive
dofa_ckpt = f'{DRIVE}/DOFA_ViT_base_e100.pth'
if not os.path.exists(dofa_ckpt):
    os.system('cd /content/DOFA && python checkpoints/download_weights.py')
    os.system(f'cp /content/DOFA/checkpoints/DOFA_ViT_base_e100.pth "{dofa_ckpt}"')
print('repos + DOFA weights ready')


In [ ]:
# LEVIR-CD (training + in-domain eval) - via Kaggle
from google.colab import files
files.upload()   # <- pick your kaggle.json in the dialog
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d keykeylv/levir-cd-256
!unzip -q -o levir-cd-256.zip -d /content/LEVIR-CD
!ls /content/LEVIR-CD


In [ ]:
# WHU-CD - only needed for the zero-shot eval.
# wgcban's Dropbox has a DAILY bandwidth quota; if it returns a ~176KB text/html page,
# it's temporarily banned - try again after the daily reset, then cache to Drive (below).
if os.path.exists(f'{DRIVE}/WHU-CD-256.zip'):
    !cp "{DRIVE}/WHU-CD-256.zip" /content/WHU.zip          # use the cached copy if we have one
else:
    !wget -L "https://www.dropbox.com/s/r76a00jcxp5d3hl/WHU-CD-256.zip?dl=1" -O /content/WHU.zip
!ls -lh /content/WHU.zip
!unzip -q -o /content/WHU.zip -d /content/WHU-CD
# once you have a REAL ~1.8GB zip, cache it so you never fight Dropbox again:
# !cp /content/WHU.zip "{DRIVE}/WHU-CD-256.zip"
!ls /content/WHU-CD


In [ ]:
# Point BIT's data_config at both datasets
path = '/content/BIT_CD/data_config.py'
s = open(path).read()
s = s.replace("'path to the root of LEVIR-CD dataset'", "'/content/LEVIR-CD/'")
s = s.replace("elif data_name == 'quick_start':",
              "elif data_name == 'WHU':\n            self.root_dir = '/content/WHU-CD/WHU-CD-256/'\n        elif data_name == 'quick_start':")
open(path, 'w').write(s)
print('data_config patched (LEVIR + WHU)')


## 1. Build the DOFA arm  →  `bit`

Load DOFA, wrap it in an adapter that mimics BIT's ResNet output shape `[B, 32, 64, 64]`,
and swap it into BIT's `forward_single`.


In [ ]:
import torch, sys, types
import torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, '/content/DOFA'); sys.path.insert(0, '/content/BIT_CD')
from dofa_v1 import vit_base_patch16

DRIVE = '/content/drive/MyDrive'
vit_model = vit_base_patch16()
vit_model.load_state_dict(torch.load(f'{DRIVE}/DOFA_ViT_base_e100.pth', weights_only=False), strict=False)
vit_model = vit_model.cuda()
print('DOFA encoder loaded')


In [ ]:

import torch, sys, types
import torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, '/content/BIT_CD')
from models.networks import BASE_Transformer

# --- DOFA adapter: patch tokens -> ResNet-shaped feature map ---
def get_spatial_tokens(model, x, wave_list):
  waves = torch.tensor(wave_list, device=x.device).float(); model.waves = waves
  x, _ = model.patch_embed(x, model.waves)
  x = x + model.pos_embed[:, 1:, :]
  cls = (model.cls_token + model.pos_embed[:, :1, :]).expand(x.shape[0], -1, -1)
  x = torch.cat((cls, x), dim=1)
  for block in model.blocks:
      x = block(x)
  return model.fc_norm(x)[:, 1:, :]                      # [B, 196, 768] patch tokens (14x14)

squeeze = nn.Conv2d(768, 32, kernel_size=1).cuda()          # depth 768 -> 32

def dofa_backbone(image, model, squeeze, wave_list=[0.665, 0.56, 0.49]):
  image = F.interpolate(image, size=(224, 224), mode='bilinear', align_corners=False)  # DOFA pretrained at 224
  tokens = get_spatial_tokens(model, image, wave_list)
  B = image.shape[0]
  grid = tokens.reshape(B, 14, 14, 768).permute(0, 3, 1, 2)                             # [B, 768, 14, 14]
  grid = squeeze(grid)                                                                  # [B, 32, 14, 14]
  return F.interpolate(grid, size=(64, 64), mode='bilinear', align_corners=False)       # [B, 32, 64, 64]


In [ ]:
# --- build BIT (base_transformer_pos_s4_dd8) and swap in DOFA ---
from models.networks import BASE_Transformer
bit = BASE_Transformer(input_nc=3, output_nc=2, token_len=4, resnet_stages_num=4,
                       with_pos='learned', enc_depth=1, dec_depth=8).cuda()
bit.dofa = vit_model
bit.squeeze = squeeze
bit.forward_single = types.MethodType(lambda self, x: dofa_backbone(x, self.dofa, self.squeeze), bit)

# sanity check - expect torch.Size([1, 2, 256, 256])
A = torch.randn(1, 3, 256, 256).cuda(); B = torch.randn(1, 3, 256, 256).cuda()
print('BIT-on-DOFA ready:', bit(A, B).shape)


## 2. Build the DINOv2 arm -> `bit_dino`

Load DINOv2, wrap its patch-token grid in a small adapter, and swap it into BIT's `forward_single`.


In [ ]:
# load DINOv2 (auto-downloads ~330MB; 'xFormers not available' warnings are harmless)
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').cuda().eval()

# --- DINOv2 adapter ---
squeeze_dino = nn.Conv2d(768, 32, kernel_size=1).cuda()

def dino_backbone(image, model, squeeze):
  image = F.interpolate(image, size=(224, 224), mode='bilinear', align_corners=False)  # patch14 needs 224
  tokens = model.forward_features(image)['x_norm_patchtokens']   # [B, 256, 768] (already normalized)
  B = image.shape[0]
  grid = tokens.reshape(B, 16, 16, 768).permute(0, 3, 1, 2)      # [B, 768, 16, 16] (16x16 grid)
  grid = squeeze(grid)                                           # [B, 32, 16, 16]
  return F.interpolate(grid, size=(64, 64), mode='bilinear', align_corners=False)  # [B, 32, 64, 64]


In [ ]:
# --- build a SEPARATE BIT and swap in DINOv2 ---
import sys, types
sys.path.insert(0, '/content/BIT_CD')
from models.networks import BASE_Transformer

bit_dino = BASE_Transformer(input_nc=3, output_nc=2, token_len=4, resnet_stages_num=4,
                            with_pos='learned', enc_depth=1, dec_depth=8).cuda()
bit_dino.dino = dino
bit_dino.squeeze = squeeze_dino
bit_dino.forward_single = types.MethodType(lambda self, x: dino_backbone(x, self.dino, self.squeeze), bit_dino)

# sanity check - expect torch.Size([1, 2, 256, 256])
A = torch.randn(1, 3, 256, 256).cuda(); B = torch.randn(1, 3, 256, 256).cuda()
print('BIT-on-DINOv2 ready:', bit_dino(A, B).shape)


## 3. Train — run the arm(s) you need

Each cell saves to its own `.pth`, so all arms coexist. Loss typically plateaus by
~epoch 10-15 for frozen arms — watch it and stop early.


### 3a. DOFA frozen  →  `dofa_bit.pth`


In [ ]:
from utils import get_loader

for p in bit.dofa.parameters():           # freeze the backbone
  p.requires_grad = False

train_loader = get_loader(data_name='LEVIR', img_size=256, batch_size=8, is_train=True, split='train')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD([p for p in bit.parameters() if p.requires_grad], lr=0.01)

save_path = f'{DRIVE}/dofa_bit.pth'
num_epochs = 50
bit.train(); bit.dofa.eval()
for epoch in range(num_epochs):
  running = 0.0
  for batch in train_loader:
      A = batch['A'].cuda(); B = batch['B'].cuda()
      L = batch['L'].cuda().squeeze(1).long()
      loss = criterion(bit(A, B), L)
      optimizer.zero_grad(); loss.backward(); optimizer.step()
      running += loss.item()
  print(f"epoch {epoch+1}/{num_epochs}  avg_loss {running/len(train_loader):.4f}")
  torch.save(bit.state_dict(), save_path)
print("done ->", save_path)


### 3b. DOFA fine-tuned  →  `dofa_bit_finetuned.pth`  (unfreeze, AdamW, small LR)


In [ ]:
from utils import get_loader

for p in bit.dofa.parameters():           # unfreeze - DOFA adapts to change detection
  p.requires_grad = True

train_loader = get_loader(data_name='LEVIR', img_size=256, batch_size=4, is_train=True, split='train')  # smaller batch: ViT backprop
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in bit.parameters() if p.requires_grad], lr=1e-4)

save_path = f'{DRIVE}/dofa_bit_finetuned.pth'
num_epochs = 20
bit.train()
for epoch in range(num_epochs):
  running = 0.0
  for batch in train_loader:
      A = batch['A'].cuda(); B = batch['B'].cuda()
      L = batch['L'].cuda().squeeze(1).long()
      loss = criterion(bit(A, B), L)
      optimizer.zero_grad(); loss.backward(); optimizer.step()
      running += loss.item()
  print(f"epoch {epoch+1}/{num_epochs}  avg_loss {running/len(train_loader):.4f}")
  torch.save(bit.state_dict(), save_path)
print("done ->", save_path)


### 3c. DINOv2 frozen  →  `dino_bit.pth`


In [ ]:
from utils import get_loader

for p in bit_dino.dino.parameters():      # freeze DINOv2
  p.requires_grad = False

train_loader = get_loader(data_name='LEVIR', img_size=256, batch_size=8, is_train=True, split='train')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD([p for p in bit_dino.parameters() if p.requires_grad], lr=0.01)

save_path = f'{DRIVE}/dino_bit.pth'
num_epochs = 50
bit_dino.train(); bit_dino.dino.eval()
for epoch in range(num_epochs):
  running = 0.0
  for batch in train_loader:
      A = batch['A'].cuda(); B = batch['B'].cuda()
      L = batch['L'].cuda().squeeze(1).long()
      loss = criterion(bit_dino(A, B), L)
      optimizer.zero_grad(); loss.backward(); optimizer.step()
      running += loss.item()
  print(f"epoch {epoch+1}/{num_epochs}  avg_loss {running/len(train_loader):.4f}")
  torch.save(bit_dino.state_dict(), save_path)
print("done ->", save_path)


### 3d. DINOv2 fine-tuned -> `dino_bit_finetuned.pth`

Unfreeze DINOv2 and fine-tune with AdamW at a small learning rate.


In [ ]:
from utils import get_loader
for p in bit_dino.dino.parameters():      # unfreeze DINOv2
    p.requires_grad = True
train_loader = get_loader(data_name='LEVIR', img_size=256, batch_size=4, is_train=True, split='train')
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in bit_dino.parameters() if p.requires_grad], lr=1e-4)

save_path = f'{DRIVE}/dino_bit_finetuned.pth'
num_epochs = 20
bit_dino.train()
for epoch in range(num_epochs):
    running = 0.0
    for batch in train_loader:
        A = batch['A'].cuda(); B = batch['B'].cuda()
        L = batch['L'].cuda().squeeze(1).long()
        loss = criterion(bit_dino(A, B), L)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item()
    print(f"epoch {epoch+1}/{num_epochs}  avg_loss {running/len(train_loader):.4f}")
    torch.save(bit_dino.state_dict(), save_path)
print("done ->", save_path)


## WHU Cache Check

Use this only if WHU-CD is not already unpacked in the runtime. The key sanity check is that `/content/WHU.zip` should be about 1.8GB, not a tiny HTML quota page.


In [ ]:
!wget -L "https://www.dropbox.com/s/r76a00jcxp5d3hl/WHU-CD-256.zip?dl=1" -O /content/WHU.zip
!ls -lh /content/WHU.zip          # sanity: should be ~1.8G, NOT 176K
!cp /content/WHU.zip "{DRIVE}/WHU-CD-256.zip"   # <-- cache it NOW, once and for all
!unzip -q -o /content/WHU.zip -d /content/WHU-CD
!ls /content/WHU-CD/WHU-CD-256/list             # sanity: should list test.txt etc.


## 4. Evaluate

`evaluate(model, data_name)` scores any arm. Load the weights you want into the matching
model, then read the change-class metrics (`F1_1`, `iou_1`, `precision_1`, `recall_1`).


In [ ]:
from utils import get_loader
from misc.metric_tool import ConfuseMatrixMeter

def evaluate(model, data_name):
  loader = get_loader(data_name, img_size=256, batch_size=8, is_train=False, split='test')
  metric = ConfuseMatrixMeter(n_class=2); model.eval()
  with torch.no_grad():
      for batch in loader:
          A = batch['A'].cuda(); B = batch['B'].cuda()
          pred = torch.argmax(model(A, B), dim=1)
          gt = batch['L'].cuda().squeeze(1).long()
          metric.update_cm(pr=pred.cpu().numpy(), gt=gt.cpu().numpy())
  return metric.get_scores()


## Final Comparison Table

This is the clean result to show: change-class F1 on LEVIR and WHU, plus the cross-dataset robustness ratio.

`keeps = WHU F1 / LEVIR F1`.


In [ ]:
runs = [
    ("DOFA-frozen",   bit,      f'{DRIVE}/dofa_bit.pth'),
    ("DOFA-ft",       bit,      f'{DRIVE}/dofa_bit_finetuned.pth'),
    ("DINOv2-frozen", bit_dino, f'{DRIVE}/dino_bit.pth'),
    ("DINOv2-ft",     bit_dino, f'{DRIVE}/dino_bit_finetuned.pth'),
]

print(f"{'model':15} {'LEVIR':>7} {'WHU':>7} {'keeps':>7}   WHU P/R")
print("-" * 52)
for name, model, path in runs:
    model.load_state_dict(torch.load(path, weights_only=False))
    levir_scores = evaluate(model, 'LEVIR')
    whu_scores = evaluate(model, 'WHU')
    lf1 = levir_scores['F1_1']
    wf1 = whu_scores['F1_1']
    wp = whu_scores['precision_1']
    wr = whu_scores['recall_1']
    print(f"{name:15} {lf1:7.4f} {wf1:7.4f} {wf1 / lf1:6.1%}   P{wp:.3f}/R{wr:.3f}")


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


model             LEVIR     WHU   keeps   WHU P/R
----------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


DOFA-frozen      0.7208  0.6926  96.1%   P0.789/R0.617
DOFA-ft          0.7244  0.5181  71.5%   P0.741/R0.398
DINOv2-frozen    0.7729  0.7805 101.0%   P0.877/R0.703
DINOv2-ft        0.8031  0.3862  48.1%   P0.266/R0.704


## Reading the Result

- DINOv2-frozen is the strongest cross-dataset model in this run: WHU F1 is slightly above its LEVIR F1.
- Fine-tuning DINOv2 improves LEVIR F1 but collapses on WHU, mainly through precision loss.
- DOFA-frozen is robust but lower than DINOv2-frozen here.
- DOFA fine-tuning also hurts cross-dataset transfer.

Careful claim: this is a single dataset-pair/single-seed result. It is useful evidence for year-over-year property comparison because it tests exactly the kind of domain shift expected across location, year, image source, and visual conditions.
